# 20_multimodal: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/20_multimodal/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Score genuine image and text vectors in the same SigLIP space.

Run: uv run  python modules/20_multimodal/examples/01_from_scratch.py
Expected: a ranked image gallery for a text query.
CPU: seconds to one minute. Needs: base, prepared image vectors. No implicit downloads.
"""

from common.apps.artifacts import load_artifact
from common.embeddings import normalize


def main() -> None:
    ids, images, model = load_artifact("image_gallery")
    _, queries, query_model = load_artifact("image_queries")
    assert model["space"] == query_model["space"]
    # KEY IDEA: normalize before a cosine dot product. MiniLM is a different space.
    scores = normalize(images) @ normalize(queries[0])
    ranking = sorted(range(len(ids)), key=lambda i: (-float(scores[i]), ids[i]))[:5]
    print([(ids[i], float(scores[i])) for i in ranking])


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## 02_production

Source: `modules/20_multimodal/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("qdrant_client") is None:
    print("SKIP native comparison: --extra qdrant")
else:
    """Store image and text vectors together in one Qdrant point.

    Run: uv run --extra qdrant python modules/20_multimodal/examples/02_production.py
    Expected: same payload, two named vector spaces, cross-modal results.
    CPU: seconds to one minute. Needs: qdrant extra, prepared SigLIP vectors. No implicit downloads.
    """
    # course-extra: qdrant

    import time

    from common.apps.artifacts import load_artifact
    from common.apps.database import AppDatabase
    from common.pretty import show_hits
    from common.tier4 import open_photos

    def main() -> None:
        with AppDatabase("qdrant-local") as db:
            photos, _, _ = open_photos(db)
            try:
                _, queries, meta = load_artifact("image_queries")
                info = db.client.get_collection(photos.name)
                assert set(info.config.params.vectors) == {"image", "text"}
                start = time.perf_counter()
                # KEY IDEA: the query must declare its model space despite identical dimensions.
                hits = photos.search(
                    queries[0], space=meta["space"], image_weight=1, exclude=["p0000000"]
                )
                show_hits(hits, (time.perf_counter() - start) * 1000, title="Text to image")
            finally:
                photos.close()

    if __name__ == "__main__":
        from common.errors import run_cli

        run_cli(main)

## 03_measure

Source: `modules/20_multimodal/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Evaluate text-to-image, photo-to-image and weighted fusion.

Run: uv run  python modules/20_multimodal/examples/03_measure.py
Expected: held-out nDCG for each mode with self-image excluded.
CPU: seconds to one minute. Needs: base, frozen image/text artifacts. No implicit downloads.
"""

from common.apps.database import AppDatabase
from common.apps.evaluation import photo_evaluation
from common.teaching import evidence


def main() -> None:
    with AppDatabase() as db:
        # KEY IDEA: remove the query image ID before measuring relevance.
        result = photo_evaluation(db)
    print(result["held_out"])
    print(result["scope"])
    evidence("photo-search", result, tier=4)


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## 04_fusion

Source: `modules/20_multimodal/examples/04_fusion.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Inspect weighted rank fusion without adding incompatible raw scores.

Run: uv run  python modules/20_multimodal/examples/04_fusion.py
Expected: changing weights changes which item ranks first.
CPU: seconds to one minute. Needs: base, NumPy. No implicit downloads.
"""

from common.apps.multimodal import weighted_fusion


def main() -> None:
    ranks = {"image": ["red-mug", "blue-mug"], "text": ["blue-mug", "red-mug"]}
    for weight in [0.0, 0.5, 1.0]:
        # KEY IDEA: ranks have a common interpretation; raw model scores need calibration.
        print(weight, weighted_fusion(ranks, {"image": weight, "text": 1 - weight}))
    assert weighted_fusion(ranks, {"image": 1, "text": 0})[0][0] == "red-mug"


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
from common.apps.multimodal import weighted_fusion

print(weighted_fusion({"image": ["a", "b"], "text": ["b", "a"]}, {"image": 0.7, "text": 0.3}))

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.